In [1]:
!pip -q install google-genai

from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Missing GEMINI_API_KEY in Colab Secrets.")

client = genai.Client(api_key=api_key)

model = "gemini-flash-latest"

def call_llm(prompt):
    response = client.models.generate_content(
        model=model,
        contents=prompt
    )
    return response.text

In [2]:
react_prompt = """
You must follow this exact ReACT structure.

STAGE 1 — REASON/PLAN:
Briefly explain your plan in 3–5 bullet points.

STAGE 2 — ACT (CODE):
Output ONLY Python code inside a ```python``` code block.

Task:
Simulate subscription revenue for 12 months.

Assumptions:
- Start with 200 paying users
- Paid users grow 5% per month
- ARPU = $15 per month

Requirements:
- Use pandas
- Output a table with columns:
  month, paid_users, monthly_revenue, cumulative_revenue
- Users must be integers
- Revenue must be rounded to 2 decimals
"""

response = call_llm(react_prompt)

print(response)

STAGE 1 — REASON/PLAN:
*   Define the initial simulation parameters, including the starting user base, the monthly growth percentage, and the average revenue per user (ARPU).
*   Generate the monthly user counts and revenue figures by iterating through a 12-month timeline using a growth formula.
*   Construct a pandas DataFrame to organize the data, ensuring user counts are integers and revenue totals are formatted to two decimal places.

STAGE 2 — ACT (CODE):
```python
import pandas as pd

# Constants
initial_users = 200
growth_rate = 0.05
arpu = 15
months = 12

data = []
current_users = initial_users
cumulative_revenue = 0.0

for month in range(1, months + 1):
    # Calculate metrics
    monthly_revenue = current_users * arpu
    cumulative_revenue += monthly_revenue
    
    # Store data
    data.append({
        "month": month,
        "paid_users": int(current_users),
        "monthly_revenue": round(float(monthly_revenue), 2),
        "cumulative_revenue": round(float(cumulative_

In [3]:
import re

# Extract python code block
match = re.search(r"```python\s*(.*?)```", response, re.DOTALL)

if not match:
    raise ValueError("No python code block found in model output.")

generated_code = match.group(1)

print("----- GENERATED CODE -----\n")
print(generated_code)

print("\n----- EXECUTION OUTPUT -----\n")

# Execute the generated code
exec(generated_code)

----- GENERATED CODE -----

import pandas as pd

# Constants
initial_users = 200
growth_rate = 0.05
arpu = 15
months = 12

data = []
current_users = initial_users
cumulative_revenue = 0.0

for month in range(1, months + 1):
    # Calculate metrics
    monthly_revenue = current_users * arpu
    cumulative_revenue += monthly_revenue
    
    # Store data
    data.append({
        "month": month,
        "paid_users": int(current_users),
        "monthly_revenue": round(float(monthly_revenue), 2),
        "cumulative_revenue": round(float(cumulative_revenue), 2)
    })
    
    # Update users for next month (compounded growth)
    current_users = current_users * (1 + growth_rate)

# Create DataFrame
df = pd.DataFrame(data)

# Display results
print(df.to_string(index=False))


----- EXECUTION OUTPUT -----

 month  paid_users  monthly_revenue  cumulative_revenue
     1         200          3000.00             3000.00
     2         210          3150.00             6150.00
     3         220

In [4]:
# ReACT Observation + Improvement Stage

fix_prompt = """
You previously generated Python code to simulate subscription revenue.

Now improve the code by:
1. Adding a check to ensure paid users never drop below 0.
2. Adding a simple try/except block for safety.
3. Adding a final print statement summarizing total revenue after 12 months.

Output ONLY revised Python code inside a ```python``` block.
"""

improved_response = call_llm(fix_prompt)

print(improved_response)

```python
def simulate_subscription_revenue():
    try:
        # Initial Parameters
        initial_users = 100
        monthly_growth = 25
        churn_rate = 0.10  # 10% churn
        price_per_month = 20.0
        months = 12

        current_users = initial_users
        total_revenue = 0.0

        print(f"{'Month':<10} | {'Users':<10} | {'Monthly Revenue':<15}")
        print("-" * 40)

        for month in range(1, months + 1):
            # Calculate churned users and new additions
            churned_users = int(current_users * churn_rate)
            current_users = (current_users + monthly_growth) - churned_users

            # Safety Check: Ensure users never drop below 0
            if current_users < 0:
                current_users = 0

            # Calculate revenue for the month
            monthly_revenue = current_users * price_per_month
            total_revenue += monthly_revenue

            print(f"{month:<10} | {current_users:<10} | ${monthly_revenue:,.2f}")


In [5]:
import re

m2 = re.search(r"```python\s*(.*?)```", improved_response, re.DOTALL)
if not m2:
    raise ValueError("No python code block found in improved output.")

improved_code = m2.group(1)

print("----- IMPROVED CODE -----\n")
print(improved_code)

print("\n----- IMPROVED EXECUTION OUTPUT -----\n")
exec(improved_code)

----- IMPROVED CODE -----

def simulate_subscription_revenue():
    try:
        # Initial Parameters
        initial_users = 100
        monthly_growth = 25
        churn_rate = 0.10  # 10% churn
        price_per_month = 20.0
        months = 12

        current_users = initial_users
        total_revenue = 0.0

        print(f"{'Month':<10} | {'Users':<10} | {'Monthly Revenue':<15}")
        print("-" * 40)

        for month in range(1, months + 1):
            # Calculate churned users and new additions
            churned_users = int(current_users * churn_rate)
            current_users = (current_users + monthly_growth) - churned_users

            # Safety Check: Ensure users never drop below 0
            if current_users < 0:
                current_users = 0

            # Calculate revenue for the month
            monthly_revenue = current_users * price_per_month
            total_revenue += monthly_revenue

            print(f"{month:<10} | {current_users:<10} | ${monthly